In [2]:
import torch 

x=torch.ones(5)
y=torch.zeros(3)
w=torch.randn(5,3,requires_grad=True) #生成标准动态分布
b=torch.randn(3,requires_grad=True)
z=torch.matmul(x,w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)# 这个是sigmoid函数和loss函数的直接复合

In [3]:
print(f"Gradient function for z = {z.grad_fn}")# 看一下这个产生的算法（这个是线性）
print(f"Gradient function for loss = {loss.grad_fn}")# 这个是sigmoid和loss的复合

Gradient function for z = <AddBackward0 object at 0x000002AE1F2ED090>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x000002AE1F2EE950>


In [4]:
loss.backward()
print(w)
print(b)

tensor([[-0.7112, -0.4981, -1.2544],
        [ 1.3711,  0.0913,  0.1945],
        [ 1.1327,  0.3769, -0.3532],
        [-0.0047,  2.1870, -0.3768],
        [-0.4670,  0.1234, -1.4020]], requires_grad=True)
tensor([ 0.1061, -0.0080, -0.0072], requires_grad=True)


In [5]:
z_test=torch.matmul(x,w)+b
print(z_test.requires_grad)

with torch.no_grad():
    z_test=z_test=torch.matmul(x,w)+b

print(z_test.requires_grad)# 这里变成False说明z的requires_grad已经被更改了，但是若是在循环中，因为z其实是临时变量，所以会重新可要求导

True
False


In [6]:
z_det = z_test.detach()
print(z_det.requires_grad)

False


In [7]:
# 实质上pytorch中自动求导是记录了每一步，在反向传播的时候会依次返回并销毁，保证每次都按照当时的状态保存

In [8]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()# 直接全部清空了
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])
